# Phase 6.5 shard 12 (forest65)

Runs **108 cells** of the frozen Phase 6.5 manifest (`G3-PHASE65-v1`), covering: `causal_drf`, `causal_drf_log`, `causal_drf_retn`, `drf`, `drf_log`.

This shard runs the R forest baselines, including the two adversarial controls (log geometry and bandwidth retune). The setup cell installs R, the pinned `drf` 1.3.1, and the authors' causal-clean package at the frozen commit; fifteen to twenty-five minutes.

Estimated single-threaded compute on the reference machine is about **87 minutes**. Colab cores are slower, so allow two to three times that, plus any install time above. This fits comfortably inside a nine hour session.

**Run every cell in order.** The last cell downloads a `.zip`; collect every shard's zip into `results/phase65/colab_shards/` (logs into `results/manifests/`) and run `python research/run_phase65.py merge`.


In [ ]:
# Thread pinning MUST happen before NumPy or SciPy are imported.
# OpenMP sizes its pool at initialisation, so setting these
# afterwards is silently ineffective.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Clone the repository at the pinned commit

Remote `https://github.com/hugogobato/wasserstein-causal-forests.git`, commit `bfe99cb3f305`. After checkout the notebook asserts the frozen manifest checksum, so a clone of anything but the generating commit fails here rather than mid-run.

In [ ]:
import subprocess, pathlib, os, sys, json, hashlib

REPO = 'https://github.com/hugogobato/wasserstein-causal-forests.git'
COMMIT = 'bfe99cb3f305d5f9372dfb723888ed128b8f6ed9'
EXPECTED_CHECKSUM = '4e28d308ca99cde4c81379524fc4492a15b38f029b449899b0a307b6c0ace110'

workdir = pathlib.Path('/content/wcf')
if not workdir.exists():
    subprocess.run(['git', 'init', '-q', str(workdir)], check=True)
    subprocess.run(
        ['git', '-C', str(workdir), 'remote', 'add', 'origin', REPO],
        check=True,
    )
# A shallow fetch of the exact commit: nothing else is downloaded.
    subprocess.run(
        ['git', '-C', str(workdir), 'fetch', '-q', '--depth', '1',
         'origin', COMMIT], check=True,
    )
    subprocess.run(
        ['git', '-C', str(workdir), 'checkout', '-q', 'FETCH_HEAD'],
        check=True,
    )
os.chdir(workdir)
sys.path.insert(0, str(workdir / 'src'))
os.environ['WCF_CAUSAL_DRF_R_LIB'] = '/content/wcf/results/Rlib/causal_drf'

manifest = json.load(open(
    'results/manifests/phase65_manifest.json', encoding='utf-8'
))
checksum = hashlib.sha256(
    json.dumps(manifest['cells'], sort_keys=True).encode('utf-8')
).hexdigest()
assert checksum == EXPECTED_CHECKSUM, (
    'the cloned manifest does not match the frozen grid: '
    f'{checksum} != {EXPECTED_CHECKSUM}'
)
print('repo ready at commit ' + COMMIT[:12] + '; '
      + str(manifest['n_cells']) + ' frozen cells verified')

## 2. Dependencies

In [ ]:
# This group runs the R forest baselines, including Causal-DRF
# through the authors' causal-clean package at the frozen commit.
# The causal-clean repository is a monorepo whose R package sits in
# r-package/drf, so the installer fetches the exact-commit tarball
# from codeload (no GitHub API, hence no shared-IP rate limit) and
# runs R CMD INSTALL on that subdirectory. Expect fifteen to twenty-
# five minutes for this cell.
%%bash
set -e
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev curl > /dev/null 2>&1
Rscript -e 'options(Ncpus=2); install.packages(c("Rcpp","RcppEigen","jsonlite","remotes","transport","fastDummies","kernlab"), repos="https://cloud.r-project.org", quiet=TRUE)'
# CRAN drf 1.3.1 drives the paper-DRF and W-DRF-T drivers; the
# causal-clean library below shadows it only for Causal-DRF cells.
Rscript -e 'options(Ncpus=2); if (!requireNamespace("drf", quietly=TRUE)) install.packages("drf", repos="https://cloud.r-project.org", quiet=TRUE); cat("CRAN drf", as.character(packageVersion("drf")), "ready\n")'
mkdir -p results/Rlib/causal_drf
CAUSAL_SHA="0a1a508444176b5b1553f13e832be93a374b0af2"
if [ ! -d results/Rlib/causal_drf/drf ]; then
  TARBALL="/tmp/causal_clean_${CAUSAL_SHA:0:12}.tar.gz"
  curl -sL "https://codeload.github.com/herbps10/drf/tar.gz/${CAUSAL_SHA}" -o "$TARBALL"
  EXTRACT="/tmp/causal_clean_src"
  rm -rf "$EXTRACT"; mkdir -p "$EXTRACT"
  tar -xzf "$TARBALL" -C "$EXTRACT"
  PKG_DIR=$(find "$EXTRACT" -maxdepth 3 -type d -path "*r-package/drf" | head -1)
  echo "installing causal-clean drf from $PKG_DIR"
  R CMD INSTALL --library=results/Rlib/causal_drf "$PKG_DIR" \
    || Rscript -e 'options(Ncpus=2); .libPaths(c("results/Rlib/causal_drf",.libPaths())); remotes::install_github("herbps10/drf", ref="0a1a508444176b5b1553f13e832be93a374b0af2", subdir="r-package/drf", lib="results/Rlib/causal_drf", upgrade="never", quiet=TRUE)'
fi
Rscript -e '.libPaths(c("results/Rlib/causal_drf",.libPaths())); stopifnot(requireNamespace("drf", quietly=TRUE)); cat("causal-clean drf", as.character(packageVersion("drf")), "ready\n")'
echo 'setup complete'


## 3. This shard's cells

In [ ]:
import json, collections
SHARD_INDEX = 12
CELLS = json.loads('''[{"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "e0215514b3ebe13a", "test_seed": 900000}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "f235a0c45357e374", "test_seed": 900000}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "b6a7862266dc0076", "test_seed": 900000}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "993968cb98edcb31", "test_seed": 900000}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "40f48b7808845708", "test_seed": 900000}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "3e9bfc871a27525a", "test_seed": 900000}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "972f5419da3f6a24", "test_seed": 900000}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "62ac8fce099b0834", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 0, "cell_key": "f58edd5dac11e9b8", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 5, "cell_key": "78408a552868a36a", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 0, "cell_key": "bd2f603eb0295cca", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 5, "cell_key": "8f5d2d94b7713a43", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 0, "cell_key": "7178882054d824f2", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 5, "cell_key": "769d518ef6fa5521", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 0, "cell_key": "45bc6eb91bd8a0a9", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 5, "cell_key": "235061767f90cc6f", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 0, "cell_key": "dc61d8bdf8362a7c", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 5, "cell_key": "621c71d55ef91328", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 0, "cell_key": "d6e899e0b7c1631a", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 5, "cell_key": "42d696322a03c8bd", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 0, "cell_key": "586e9c27060826dd", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 5, "cell_key": "a8bad4d2fa65cf5f", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 0, "cell_key": "d2355594da1368b8", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 5, "cell_key": "a0d8c1e26117bfc2", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 0, "cell_key": "c04627bbfc9d5054", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 5, "cell_key": "2fb9b5d3d2443041", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 0, "cell_key": "6af8312dd1ceda97", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 5, "cell_key": "48256a062381b584", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 0, "cell_key": "c8e8a37afd964887", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 5, "cell_key": "2528dab3cf7549ba", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 0, "cell_key": "69a8d7112759937b", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 5, "cell_key": "a0d33b6daf60ee87", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 0, "cell_key": "5f305407b63a23e1", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 5, "cell_key": "7ac71054cdc8ad99", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 0, "cell_key": "1c9b14f72dad9b73", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 5, "cell_key": "65c23ec29cc59342", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 0, "cell_key": "86a78bc861d6379f", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 5, "cell_key": "de244d1b6ca7b6a7", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 0, "cell_key": "a2f2c5983ee47aa0", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 5, "cell_key": "4083ef9e14402c3c", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "b3b9a17c974ed625", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "f632156fba83f408", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "468bd4cb5aea3791", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "4626cd779142058b", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "840410a60137644d", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "f70139c2f5f8e8cd", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "cad430e55d818ace", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "216951239fe5ff8a", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "0d215fb48e0042d2", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "a2a07b290f96829e", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "3db48c75649817fd", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "aa151ca008770ba1", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "206eeff4e1e65dbf", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "699701cbcb76b5dd", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "cced8eb083309adc", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "e9556ec5517c0a4e", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "e0d3c0edac007e11", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "5c29698284122fad", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "ff868e16e51dd5fe", "test_seed": 900000}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "522c380d9c375a2d", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "84a29581d0440415", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "97c48ab2f95111f3", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "777ff191e342d919", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "11e7174e11b2dfce", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "29f68a4bd7221401", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "f767b8173a98bcad", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "7b36e06d8378e3bc", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "afe8f5b4ed69cbbb", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "1364e29502f2172e", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "b6cb789a96298348", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "8ec96eba7e351677", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "58483b91245bc657", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "bc018d11be23d2a1", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "e158606620d40363", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "48bb69e26a2ee615", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "c7a8adec48ae684d", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 0, "cell_key": "82d4ff4f622b7fc0", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 5, "cell_key": "3f201a09ad081a2f", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 0, "cell_key": "e4a7e310e141641c", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 5, "cell_key": "e3a2bcb57308f384", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 0, "cell_key": "41688ebb4a9e870f", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 5, "cell_key": "ed833711bdb18b9f", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 0, "cell_key": "0c5b7a54d87ea265", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 5, "cell_key": "eceefa7ccad040e4", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 0, "cell_key": "6b1827783e70a127", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 5, "cell_key": "031ef00970d50720", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 0, "cell_key": "92b2f8f2da630fe3", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 5, "cell_key": "da756195c0f8c08b", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 0, "cell_key": "c75b21969a29c527", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 5, "cell_key": "042f328a1e0da0f2", "test_seed": 900005}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 0, "cell_key": "b8b12ef260db43fe", "test_seed": 900000}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 5, "cell_key": "59eb84d3ca639dba", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "ae33be586126d366", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "48454cbc896bf839", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "5111de8bba8500d0", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "7b023faf99dc930d", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "3ab6bb36f49d8729", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "c41d6cd0c440cb0a", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "17eeebe617fe4dec", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "e3205fca6b494b0d", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "d8f6ee9b8359e62b", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "19fe0e193e7405fa", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "38dc1352cb6f7e3b", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "97bef9cf265c6656", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 0, "cell_key": "38df594000742865", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "fc380461731f5512", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 0, "cell_key": "0a0c8873a6e8d5d1", "test_seed": 900000}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "b2061beeddd2e57b", "test_seed": 900005}]''')
print(f'{len(CELLS)} cells in this shard')
for key, count in sorted(collections.Counter(
        (c['grid'], c['dgp'], c['method'])
        for c in CELLS).items()):
    print(f'  {key[0]:12s} {key[1]:8s} {key[2]:18s} {count}')

## 4. Bandwidth-selection pilot (preregistered)

This shard contains `causal_drf_retn` cells, so it first runs the selection pilot on seeds 100 and 101, outside every decisive range, and freezes the multipliers document. The rule picks the candidate with the best held-out energy score; oracle truth is never read.

In [ ]:
from pathlib import Path
import json, numpy as np
from wasserstein_causal_forests.g3.dgps import build_dgp
from wasserstein_causal_forests.g3.phase65_methods import (
    BANDWIDTH_CANDIDATES, SELECTION_SEEDS, select_bandwidth_multiplier,
)

keys = sorted({(c['dgp'], c['n_train']) for c in CELLS
               if c['method'] == 'causal_drf_retn'})
multipliers = {}
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)
for dgp_name, n_train in keys:
    dgp = build_dgp(dgp_name, 25)
    best, means = select_bandwidth_multiplier(
        dgp, n_train, seeds=SELECTION_SEEDS,
        candidates=BANDWIDTH_CANDIDATES, cache_directory=cache,
    )
    multipliers[f'{dgp_name}|{n_train}'] = best
    scores = {str(k): round(v, 5) for k, v in means.items()}
    print(f'{dgp_name} n={n_train}: multiplier {best}  scores {scores}',
          flush=True)

document = {
    'rule': 'held-out energy score, pilot seeds 100 and 101, '
            'candidates ' + repr(BANDWIDTH_CANDIDATES),
    'multipliers': multipliers,
}
path = Path('/content/wcf/results/manifests/'
            'phase65_bandwidth_selection.json')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(document, indent=2))
print('froze', path)

## 5. Run

In [ ]:
import time
from pathlib import Path
from wasserstein_causal_forests.g3.manifest import Cell
from wasserstein_causal_forests.g3.runner import run_shard

cells = [Cell(**{k: v for k, v in item.items()
                 if k not in ('cell_key', 'test_seed')})
         for item in CELLS]

out = Path('/content/wcf/results/phase65/colab_shards')
out.mkdir(parents=True, exist_ok=True)
log = Path(f'/content/wcf/results/manifests/phase65_execution_log_{SHARD_INDEX:03d}.jsonl')
log.parent.mkdir(parents=True, exist_ok=True)
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)

started = time.time()
summary = run_shard(
    cells,
    out / f'shard_{SHARD_INDEX:03d}.parquet',
    cache_directory=cache,
    log_path=log,
    manifest_contract_id='G3-PHASE65-v1',
)
print(json.dumps(summary, indent=2))
print(f'elapsed {(time.time() - started) / 60:.1f} min')

## Check

Every cell must appear exactly once, as a success or as a failure. Failures are kept and reported at merge time; a seed is never silently replaced.

In [ ]:
import collections
records = [json.loads(line) for line in
           open(log, encoding='utf-8') if line.strip()]
status = collections.Counter(r['status'] for r in records)
print('cells logged:', len(records), '| expected:', len(CELLS))
print('status:', dict(status))
assert len(records) == len(CELLS), 'shard did not finish every cell'
for record in records:
    if record['status'] != 'ok':
        print('  FAILED', record['dgp'], record['method'],
              record['seed'])
slowest = sorted(records, key=lambda r: -r['wall_seconds'])[:5]
print('slowest cells:', [(r['method'], round(r['wall_seconds'], 1))
                         for r in slowest])

## Download the results

In [ ]:
import shutil
bundle = '/content/p65_shard_12_forest65'
staging = Path('/content/bundle')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copy(out / f'shard_{SHARD_INDEX:03d}.parquet', staging)
if log.exists():
    shutil.copy(log, staging)
output_file = shutil.make_archive(bundle, 'zip', staging)
print('bundle:', output_file,
      f'({os.path.getsize(output_file) / 1e6:.2f} MB)')

try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)